In [23]:
import pandas as pd
import numpy as np
from scipy.signal import correlate
from scipy.stats import pearsonr

In [4]:
wf_wikipedia = pd.read_csv('word_frequencies.csv')
wf_summaries = pd.read_csv('word_frequencies_summaries.csv')

In [9]:
common_columns = set(wf_wikipedia.columns).intersection(wf_summaries.columns)

wf_wikipedia_filtered = wf_wikipedia[list(common_columns)]
wf_summaries_filtered = wf_summaries[list(common_columns)]

In [16]:
wf_summaries_filtered

,woman,film,become,people,car,films,fire,becomes,group,time,death,end,space,home,years,shot,Unnamed: 0
0,0.004,0.017,0.000,0.000,0.000,0.003,0.003,0.000,0.000,0.004,0.000,0.000,0.004,0.003,0.000,0.005,1900
1,0.003,0.004,0.000,0.000,0.000,0.000,0.000,0.003,0.000,0.002,0.000,0.000,0.000,0.003,0.002,0.000,1910
2,0.003,0.004,0.002,0.000,0.000,0.000,0.000,0.002,0.000,0.003,0.002,0.002,0.000,0.003,0.001,0.000,1920
3,0.002,0.002,0.001,0.000,0.000,0.000,0.000,0.002,0.000,0.002,0.000,0.000,0.000,0.003,0.001,0.000,1930
4,0.002,0.002,0.000,0.000,0.000,0.000,0.000,0.002,0.000,0.003,0.002,0.001,0.000,0.003,0.000,0.000,1940
5,0.002,0.002,0.000,0.000,0.000,0.000,0.000,0.002,0.000,0.003,0.002,0.002,0.000,0.003,0.000,0.000,1950
6,0.002,0.003,0.001,0.000,0.002,0.000,0.000,0.002,0.001,0.003,0.002,0.001,0.000,0.003,0.000,0.000,1960
7,0.002,0.003,0.001,0.000,0.002,0.000,0.000,0.002,0.001,0.003,0.002,0.001,0.000,0.003,0.001,0.000,1970
8,0.002,0.003,0.001,0.000,0.002,0.000,0.000,0.002,0.002,0.003,0.002,0.001,0.000,0.003,0.002,0.000,1980
9,0.002,0.003,0.001,0.000,0.002,0.000,0.000,0.002,0.001,0.003,0.002,0.000,0.000,0.003,0.002,0.000,1990


In [17]:
wf_wikipedia_filtered.drop(['Unnamed: 0'], axis = 1)
wf_summaries_filtered.drop(['Unnamed: 0'], axis = 1)

,woman,film,become,people,car,films,fire,becomes,group,time,death,end,space,home,years,shot
0,0.004,0.017,0.000,0.000,0.000,0.003,0.003,0.000,0.000,0.004,0.000,0.000,0.004,0.003,0.000,0.005
1,0.003,0.004,0.000,0.000,0.000,0.000,0.000,0.003,0.000,0.002,0.000,0.000,0.000,0.003,0.002,0.000
2,0.003,0.004,0.002,0.000,0.000,0.000,0.000,0.002,0.000,0.003,0.002,0.002,0.000,0.003,0.001,0.000
3,0.002,0.002,0.001,0.000,0.000,0.000,0.000,0.002,0.000,0.002,0.000,0.000,0.000,0.003,0.001,0.000
4,0.002,0.002,0.000,0.000,0.000,0.000,0.000,0.002,0.000,0.003,0.002,0.001,0.000,0.003,0.000,0.000
5,0.002,0.002,0.000,0.000,0.000,0.000,0.000,0.002,0.000,0.003,0.002,0.002,0.000,0.003,0.000,0.000
6,0.002,0.003,0.001,0.000,0.002,0.000,0.000,0.002,0.001,0.003,0.002,0.001,0.000,0.003,0.000,0.000
7,0.002,0.003,0.001,0.000,0.002,0.000,0.000,0.002,0.001,0.003,0.002,0.001,0.000,0.003,0.001,0.000
8,0.002,0.003,0.001,0.000,0.002,0.000,0.000,0.002,0.002,0.003,0.002,0.001,0.000,0.003,0.002,0.000
9,0.002,0.003,0.001,0.000,0.002,0.000,0.000,0.002,0.001,0.003,0.002,0.000,0.000,0.003,0.002,0.000


In [19]:
wf_wikipedia_filtered = wf_wikipedia_filtered.apply(pd.to_numeric, errors='coerce')
wf_summaries_filtered = wf_summaries_filtered.apply(pd.to_numeric, errors='coerce')
wf_wikipedia_filtered = wf_wikipedia_filtered.fillna(0)
wf_summaries_filtered = wf_summaries_filtered.fillna(0)

In [24]:
results = {}

for col in common_columns:
    series_1 = wf_wikipedia_filtered[col].values
    series_2 = wf_summaries_filtered[col].values

    lags = np.arange(-len(series_1) + 1, len(series_2))
    cross_corr = correlate(series_1, series_2, mode='full')

    optimal_lag = lags[np.argmax(cross_corr)]

    aligned_series_2 = np.roll(series_2, shift=optimal_lag)

    try:
        correlation, p_value = pearsonr(series_1, aligned_series_2)
    except ValueError:
        correlation, p_value = np.nan, np.nan

    results[col] = {
        'optimal_lag': optimal_lag,
        'correlation': correlation,
        'p_value': p_value
    }

    print(f"Colonne : {col}, Optimal lag : {optimal_lag}, Correlation : {correlation}, p-value : {p_value}")


Colonne : woman, Optimal lag : 6, Correlation : 0.8049844718999243, p-value : 0.0027979543571659926
Colonne : film, Optimal lag : 2, Correlation : 0.7111214797149096, p-value : 0.014146485984596753
Colonne : become, Optimal lag : 5, Correlation : 0.479583152331272, p-value : 0.13550829005320666
Colonne : people, Optimal lag : -3, Correlation : 0.45000000000000007, p-value : 0.16489599087950865
Colonne : car, Optimal lag : -10, Correlation : 0.34641016151377546, p-value : 0.2966650369240994
Colonne : films, Optimal lag : 2, Correlation : 1.0, p-value : 0.0
Colonne : fire, Optimal lag : 10, Correlation : 1.0, p-value : 0.0
Colonne : becomes, Optimal lag : 7, Correlation : 0.5163977794943223, p-value : 0.10388813106210158
Colonne : group, Optimal lag : -5, Correlation : 0.5590169943749476, p-value : 0.07381151280528574
Colonne : time, Optimal lag : 0, Correlation : 0.45414755311462374, p-value : 0.16055786177159503
Colonne : death, Optimal lag : -5, Correlation : 0.3649002245998808, p-val

C:\Users\annab\AppData\Local\Temp\ipykernel_15376\2240920592.py:15: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  correlation, p_value = pearsonr(series_1, aligned_series_2)
